In [ ]:
import pandas as pd
import numpy as np
import pickle
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import SGDRegressor
from sklearn.linear_model import HuberRegressor
from tqdm import tqdm
import statsmodels.api as sm
import time

In [ ]:
with open("data/characteristics_with_return.pkl", "rb") as f:
    data = pickle.load(f)

In [ ]:
data.head()

In [ ]:
def R2_cal(y, y_pred):
    r2 = 1 - np.sum((y-y_pred)**2) / np.sum(y**2)
    return r2

In [ ]:
def R2_cal(y, y_pred):
    r2 = 1 - np.sum((y - y_pred)**2) / np.sum(y**2)
    return r2

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_grad(x):
    return sigmoid(x) * (1 - sigmoid(x))

def mean_squared_error(y, t):
    return 0.5 * np.sum((y - t) ** 2)

class TwoLayerNet:
    def __init__(self, input_size, hidden_size, output_size, weight_init_std=0.01):
        # 初始化权重
        self.params = {}
        self.params['W1'] = weight_init_std * np.random.randn(input_size, hidden_size)
        self.params['b1'] = np.zeros(hidden_size)
        self.params['W2'] = weight_init_std * np.random.randn(hidden_size, output_size)
        self.params['b2'] = np.zeros(output_size)

    def predict(self, x):
        W1, W2 = self.params['W1'], self.params['W2']
        b1, b2 = self.params['b1'], self.params['b2']
    
        a1 = np.dot(x, W1) + b1
        z1 = sigmoid(a1)
        a2 = np.dot(z1, W2) + b2
        y = a2  
        
        return y
        
    def loss(self, x, t):
        y = self.predict(x)
        return mean_squared_error(y, t)
    
    def accuracy(self, x, t):
        y = self.predict(x)
        error = np.mean((y - t) ** 2)  
        
        return error
          
    def gradient(self, x, t):
        W1, W2 = self.params['W1'], self.params['W2']
        b1, b2 = self.params['b1'], self.params['b2']
        grads = {}
        
        batch_num = x.shape[0]
        
        # forward
        a1 = np.dot(x, W1) + b1
        z1 = sigmoid(a1)
        a2 = np.dot(z1, W2) + b2
        y = a2
        
        # backward
        dy = (y - t) / batch_num
        grads['W2'] = np.dot(z1.T, dy)
        grads['b2'] = np.sum(dy, axis=0)
        
        da1 = np.dot(dy, W2.T)
        dz1 = sigmoid_grad(a1) * da1
        grads['W1'] = np.dot(x.T, dz1)
        grads['b1'] = np.sum(dz1, axis=0)

        return grads

def train_model(X, y, input_size):
    start_time = time.time()
    train_size = int(0.7 * len(X))
    val_size = int(0.2 * len(X))
    
    x_train, y_train = X[:train_size], y[:train_size].reshape(-1, 1)
    x_val, y_val = X[train_size:train_size + val_size], y[train_size:train_size + val_size].reshape(-1, 1)
    x_test, y_test = X[train_size + val_size:], y[train_size + val_size:].reshape(-1, 1)

    net_work = TwoLayerNet(input_size=input_size, hidden_size=16, output_size=1)

    n_epochs = 2000
    batch_size = 100
    learning_rate = 0.02

    patience = 20
    best_model_state = None
    best_loss = float('inf')
    counter = 0

    for epoch in range(n_epochs):
        for i in range(0, train_size, batch_size):
            xi = x_train[i:i + batch_size]
            yi = y_train[i:i + batch_size]

            grad = net_work.gradient(xi, yi)
            for key in ('W1', 'b1', 'W2', 'b2'):
                net_work.params[key] -= learning_rate * grad[key]
        
        val_loss = net_work.accuracy(x_val, y_val)
        
        if val_loss < best_loss:
            best_loss = val_loss
            best_model_state = net_work.params.copy()
            counter = 0
        else:
            counter += 1
            if counter >= patience:
                break

        if epoch % 2000 == 0:
            train_loss = net_work.accuracy(x_train, y_train)
            test_loss = net_work.accuracy(x_val, y_val)
            print(f"train loss: {train_loss}, val loss {val_loss}")

    for key in ('W1', 'b1', 'W2', 'b2'):
        net_work.params[key] = best_model_state[key]
    
    return net_work, x_test, y_test


start_year_train = 1957
start_year_test = 1987
end_year_test = 2016

pred_y_list = []
true_y_list = []

for year in tqdm(range(start_year_test, end_year_test + 1), desc='Processing', unit='item'):
    start_date_train = pd.Timestamp(year=start_year_train, month=1, day=1)
    end_date_train = pd.Timestamp(year=year-13, month=12, day=31)

    start_date_val = pd.Timestamp(year=year-12, month=1, day=1)
    end_date_val = pd.Timestamp(year=year-1, month=12, day=31)
    
    start_date_test = pd.Timestamp(year=year, month=1, day=1)
    end_date_test = pd.Timestamp(year=year, month=12, day=31)
    
    print(f'start_date_train: {start_date_train.strftime("%Y-%m")}')
    print(f'end_date_train: {end_date_train.strftime("%Y-%m")}')
    print(f'start_date_val: {start_date_val.strftime("%Y-%m")}')
    print(f'end_date_val: {end_date_val.strftime("%Y-%m")}')
    print(f'start_date_test: {start_date_test.strftime("%Y-%m")}')
    print(f'end_date_test: {end_date_test.strftime("%Y-%m")}')
    
    data['DATE'] = pd.to_datetime(data['DATE'], format='%Y%m')
    
    train_data = data[(data['DATE'] >= start_date_train) & (data['DATE'] <= end_date_val)]
    test_data = data[(data['DATE'] >= start_date_test) & (data['DATE'] <= end_date_test)]
    
    X_train = train_data.drop(['permno', 'DATE', 'RET'], axis=1).values  
    y_train = train_data['RET'].values  
    X_test = test_data.drop(['permno', 'DATE', 'RET'], axis=1).values
    y_test = test_data['RET'].values
    input_size = X_train.shape[1]
    
    net_work, x_test, y_test = train_model(X_train, y_train, input_size)
    y_pred = net_work.predict(X_test)
    
    pred_y_list.extend(y_pred)
    true_y_list.extend(y_test)
r2_score = R2_cal(np.array(true_y_list), np.array(pred_y_list))
print(f'R2 Score: {r2_score}')